In [1]:
from qiskit.providers import fake_provider
import networkx as nx
from random import randint

Here are some coupling maps.

In [ ]:
random_backend = fake_provider.GenericBackendV2(
    16,
    coupling_map=[[0, 8], [0, 14], [1, 9], [1, 10], [1, 12], [2, 10], [2, 15], [3, 4], [3, 6], [3, 7], [4, 5], [4, 9], [4, 11], [5, 12], [5, 13], [6, 12], [7, 8], [7, 13], [8, 12], [9, 13], [9, 15], [10, 12], [10, 13], [11, 12], [11, 14], [11, 15], [12, 13], [12, 14], [13, 14], [14, 15]]
    )

square_backend = fake_provider.GenericBackendV2(
    22,
    coupling_map=[[0, 1], [1, 2], [2, 3], [3, 4], [5, 6], [6, 7], [7, 8], [8, 9], [10, 11], [11, 12], [12, 13], [13, 14], [15, 16], [16, 17], [17, 18], [18, 19], [0, 5], [1, 6], [2, 7], [3, 8], [4, 9], [5, 10], [6, 11], [7, 12], [8, 13], [9, 14], [10, 21], [21,11], [21, 15], [11, 16], [12, 17], [13, 18], [13,20], [14, 20], [20,19]]
    )

nighthawk_backend = fake_provider.GenericBackendV2(
    120,
    coupling_map=[[0, 1], [1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9], [9, 10], [10, 11], [12, 13], [13, 14], [14, 15], [15, 16], [16, 17], [17, 18], [18, 19], [19, 20], [20, 21], [21, 22], [22, 23], [24, 25], [25, 26], [26, 27], [27, 28], [28, 29], [29, 30], [30, 31], [31, 32], [32, 33], [33, 34], [34, 35], [36, 37], [37, 38], [38, 39], [39, 40], [40, 41], [41, 42], [42, 43], [43, 44], [44, 45], [45, 46], [46, 47], [48, 49], [49, 50], [50, 51], [51, 52], [52, 53], [53, 54], [54, 55], [55, 56], [56, 57], [57, 58], [58, 59], [60, 61], [61, 62], [62, 63], [63, 64], [64, 65], [65, 66], [66, 67], [67, 68], [68, 69], [69, 70], [70, 71], [72, 73], [73, 74], [74, 75], [75, 76], [76, 77], [77, 78], [78, 79], [79, 80], [80, 81], [81, 82], [82, 83], [84, 85], [85, 86], [86, 87], [87, 88], [88, 89], [89, 90], [90, 91], [91, 92], [92, 93], [93, 94], [94, 95], [96, 97], [97, 98], [98, 99], [99, 100], [100, 101], [101, 102], [102, 103], [103, 104], [104, 105], [105, 106], [106, 107], [108, 109], [109, 110], [110, 111], [111, 112], [112, 113], [113, 114], [114, 115], [115, 116], [116, 117], [117, 118], [118, 119], [0, 12], [1, 13], [2, 14], [3, 15], [4, 16], [5, 17], [6, 18], [7, 19], [8, 20], [9, 21], [10, 22], [11, 23], [12, 24], [13, 25], [14, 26], [15, 27], [16, 28], [17, 29], [18, 30], [19, 31], [20, 32], [21, 33], [22, 34], [23, 35], [24, 36], [25, 37], [26, 38], [27, 39], [28, 40], [29, 41], [30, 42], [31, 43], [32, 44], [33, 45], [34, 46], [35, 47], [36, 48], [37, 49], [38, 50], [39, 51], [40, 52], [41, 53], [42, 54], [43, 55], [44, 56], [45, 57], [46, 58], [47, 59], [48, 60], [49, 61], [50, 62], [51, 63], [52, 64], [53, 65], [54, 66], [55, 67], [56, 68], [57, 69], [58, 70], [59, 71], [60, 72], [61, 73], [62, 74], [63, 75], [64, 76], [65, 77], [66, 78], [67, 79], [68, 80], [69, 81], [70, 82], [71, 83], [72, 84], [73, 85], [74, 86], [75, 87], [76, 88], [77, 89], [78, 90], [79, 91], [80, 92], [81, 93], [82, 94], [83, 95], [84, 96], [85, 97], [86, 98], [87, 99], [88, 100], [89, 101], [90, 102], [91, 103], [92, 104], [93, 105], [94, 106], [95, 107], [96, 108], [97, 109], [98, 110], [99, 111], [100, 112], [101, 113], [102, 114], [103, 115], [104, 116], [105, 117], [106, 118], [107, 119]]
    )

Choose one, and then let's take a look.

In [ ]:
backend = nighthawk_backend

coupling_map = backend.coupling_map

G = nx.Graph()
edges = coupling_map.get_edges()
for u, v in edges:
    G.add_edge(u, v)
nx.draw(G, with_labels=True)

Now here's a function that takes the graph corresponding to a coupling map, identifies the left and right edges and then adds in extra nodes to connect them around the back.

In [ ]:
def add_edge_nodes(G, weight=100):
    # given the graph extracted from a coupling map, add extra nodes if it has four corners
    # modifies G in place, and returns the indices of the two new nodes: nl and nr
    if len(G.nodes)%2:
        print('This coupling map has an odd number of qubits, and so is incompatible.')
    else:
        corners = []
        for n in G.nodes:
            if G.degree(n)==2:
                corners.append(n)
        if len(corners)!=4:
            print('This coupling map does not have four corners, and so is incompatible.')
        else:
            # get distances between all four corners
            corners_dist = {}
            for j in range(4):
                for k in range(j+1, 4):
                    corners_dist[corners[j], corners[k]] = nx.dijkstra_path_length(G, corners[j], corners[k])
            # find the pair of corners that are maximally distant and then look at how distant all corners are from one of these
            corner_dist = {ns:d for ns,d in corners_dist.items() if max(corners_dist, key=corners_dist.get)[0] in ns}
            # identify the closest pair of these corners as the left edge
            left_corners = min(corner_dist, key=corner_dist.get)
            # and the other as the right edge
            right_corners = tuple(n for n in corners if n not in left_corners)
            # now we can get the nodes along the sides
            left_nodes = nx.dijkstra_path(G, left_corners[0], left_corners[1])
            right_nodes = nx.dijkstra_path(G, right_corners[0], right_corners[1])
            # now we add two extra nodes to the graph
            nl = max(G.nodes)+1
            G.add_edge(nl, nl+1, weight=1.3*weight) # interval chosen to get approx 50/50 prob of this link on nighthawk
            for n in left_nodes:
                G.add_edge(nl, n, weight=randint(1,weight))
            for n in right_nodes:
                G.add_edge(nl+1, n, weight=randint(1,weight))
            return nl, nl+1

Let's add in the extra nodes and take a look.

In [ ]:
nl, nr = add_edge_nodes(G)
nx.draw(G, with_labels=True)


Now let's try many samples (with random edges) to see what fraction fully matches edges on the bulk (meaning that it pairs the two extra edges).

In [ ]:
num_samples = 1000

num = 0
for _ in range(num_samples):

    coupling_map = backend.coupling_map

    G = nx.Graph()
    edges = coupling_map.get_edges()
    for u, v in edges:
        w = randint(1,100)
        G.add_edge(u, v, weight=w)

    nl, nr = add_edge_nodes(G)
    matching = nx.algorithms.matching.max_weight_matching(G, maxcardinality=True)

    num += (nl,nr) in matching or (nr, nl) in matching

print(num/num_samples)